# Testing with Lance DB , vector embedding and Mockdata

### Källor
https://docs.lancedb.com/embedding/quickstart

https://www.sbert.net/

https://www.sbert.net/docs/sentence_transformer/pretrained_models.html
https://huggingface.co/models?library=sentence-transformers

#### Step 1: Import Required Libraries

- lancedb: The main database connection and operations
- LanceModel: Pydantic model for defining table schemas
- Vector: Field type for storing vector embeddings
- get_registry(): Access to the embedding function registry. It has all the supported as well as custom embedding functions registered by the user

In [4]:
import lancedb 
from lancedb.pydantic import LanceModel, Vector
from lancedb.embeddings import get_registry

#### Step 2: Connect to LanceDB

- database locally: uri is the name of the searchroot on the folder where lancedb will save the files. 

In [5]:
db = lancedb.connect(uri="vector_db_test")

#### Step 3: Initialize the Embedding Function
- Choose and configure your embedding model:

This creates an embedding function from the local embedding registry. The Python snippet uses the sentence-transformers provider with the BGE model; the TypeScript snippet uses the Transformers-backed huggingface provider

In [8]:
model = get_registry().get(
    "sentence-transformers").create(name="paraphrase-multilingual-MiniLM-L12-v2")

#### Step 4: Define Your Schema

- Create a Pydantic model that defines your table structure:
- SourceField(): This field will be embedded
- VectorField(): This stores the embeddings
- model.ndims(): Sets vector dimensions for your model

In [9]:
import pandas as pd

df = pd.read_csv("mock.csv")

In [ ]:
#create a textcolumn from columns in dataset for better search
df["search_text"] = (
    "Artist: " + df["artist"] +
    ", Låt: " + df["song_title"] +
    ", Genre: " + df["genre"] +
    ". " + df["description"]
)


class Tracks(LanceModel):
    artist_id: int
    artist: str
    song_id: int
    song_title: str
    genre: str
    release_year: int
    country: str
    description: str

    # search vector is created from the text that is put togetgher search text
    search_text: str = model.SourceField()
    vector: Vector(model.ndims()) = model.VectorField()

/Users/mirandalyng/Desktop/Data_engineer/GitHub_repos/examensarbete/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1105 15:16:50.555000 75918 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8057.02it/s]


#### Step 5: Create Table and Ingest Data


In [11]:
table = db.create_table("tracks", schema=Tracks, data=df, mode="overwrite")

[2026-11-05T14:17:40Z WARN  lance::dataset::write::insert] No existing dataset at /Users/mirandalyng/Desktop/Data_engineer/GitHub_repos/examensarbete/eda/vector_db_test/tracks.lance, it will be created


## test med AI -hjälp

In [12]:


# 1. Anslut till din databas-mapp
db = lancedb.connect("vector_db_test")  # Ändra till din mapps sökväg om den heter något annat

# 2. Lista alla tabeller för att se att din tabell finns
print("Befintliga tabeller:", db.table_names())

# 3. Öppna tabellen och kontrollera antal rader
table = db.open_table("tracks")
print(f"Antal rader i tabellen: {len(table)}")

Befintliga tabeller: ['tracks']
Antal rader i tabellen: 10


/var/folders/49/f5021v_n7cb6ghg2_c0drpn00000gn/T/ipykernel_75918/1635612144.py:5: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  print("Befintliga tabeller:", db.table_names())


In [13]:
# Sök efter något som beskriver innehållet i din data
query = "en energisk låt att dansa till"

# Kör sökningen och hämta de 3 bästa träffarna som en Pandas DataFrame
results = table.search(query).limit(3).to_pandas()

# Visa de mest relevanta kolumnerna samt avståndet (_distance = hur nära träffen är)
results[["artist", "song_title", "description", "_distance"]]

,artist,song_title,description,_distance
0,ABBA,Dancing Queen,Glittrig discolåt om att dansa och känna sig f...,0.755409
1,Robyn,Dancing On My Own,Känslosam danslåt om att se någon man älskar m...,0.880119
2,Avicii,Wake Me Up,Dansmusik som blandar folkinspirerad gitarr me...,0.917109


In [14]:
# Testa att filtrera på metadata samtidigt som du gör en vektorsökning
filtered_results = (
    table.search("pop")
    .where("release_year > 2000")  # SQL-filter
    .limit(5)
    .to_pandas()
)

filtered_results[["artist", "song_title", "release_year", "_distance"]]

,artist,song_title,release_year,_distance
0,Adele,Hello,2015,1.230618
1,Billie Eilish,bad guy,2019,1.280080
2,Daft Punk,Get Lucky,2013,1.334910
3,Robyn,Dancing On My Own,2010,1.402700
4,Håkan Hellström,Känn ingen sorg för mig Göteborg,2005,1.454543
